# End-to-end pipeline, step by step

One step per section, reviewed before the next is added.

## Step 1 — the small program

Two CNOTs on four logical qubits: prepare $q_0..q_3$ in $|0\rangle$,
apply `cx q0,q1` and `cx q2,q3`, measure all four in Z. Each logical
qubit is a distance-3 rotated surface-code patch, and each CNOT is
performed as a lattice-surgery merge/split.

It is written in QLX's MLIR dialect — the form QLX's compiler ingests.
`lqbit` allocates a logical qubit in region `%Cs0`; the `fabric.code`
declaration tells the compiler each logical qubit is a d=3 patch
(9 data qubits, 4 X-checks, 4 Z-checks, 3 stabilizer rounds per
logical timestep).

In [1]:
PROGRAM = """
.version 1.0
.target qlx-c
.region %Cs0, code="surface_demo", role=compute;
.entry surface_demo_entry() {
    lqbit %q0 : %Cs0;
    lqbit %q1 : %Cs0;
    lqbit %q2 : %Cs0;
    lqbit %q3 : %Cs0;
    lbit %m0;
    lbit %m1;
    lbit %m2;
    lbit %m3;
    pz %q0;
    pz %q1;
    pz %q2;
    pz %q3;
    cx %q0, %q1;
    cx %q2, %q3;
    mz %m0, %q0;
    mz %m1, %q1;
    mz %m2, %q2;
    mz %m3, %q3;
    ret;
}
""".strip()

CODE_DECLARATION = """
fabric.code @surface_demo {
  distance = 3 : i64,
  partitions = {data = 9 : i64, sx = 4 : i64, sz = 4 : i64},
  stabilize_rounds = 3 : i64
}
""".strip()

print(PROGRAM)
print()
print(CODE_DECLARATION)

.version 1.0
.target qlx-c
.region %Cs0, code="surface_demo", role=compute;
.entry surface_demo_entry() {
    lqbit %q0 : %Cs0;
    lqbit %q1 : %Cs0;
    lqbit %q2 : %Cs0;
    lqbit %q3 : %Cs0;
    lbit %m0;
    lbit %m1;
    lbit %m2;
    lbit %m3;
    pz %q0;
    pz %q1;
    pz %q2;
    pz %q3;
    cx %q0, %q1;
    cx %q2, %q3;
    mz %m0, %q0;
    mz %m1, %q1;
    mz %m2, %q2;
    mz %m3, %q3;
    ret;
}

fabric.code @surface_demo {
  distance = 3 : i64,
  partitions = {data = 9 : i64, sx = 4 : i64, sz = 4 : i64},
  stabilize_rounds = 3 : i64
}


Line by line:

| line | meaning |
|------|---------|
| `lqbit %q0 : %Cs0;` | allocate logical qubit q0 — one d=3 surface-code patch (same for q1, q2, q3) |
| `lbit %m0;` | classical bit that will hold q0's measurement result |
| `pz %q0;` | prepare q0 in logical $\|0\rangle$ |
| `cx %q0, %q1;` | logical CNOT, control q0 target q1 — lattice surgery on hardware |
| `cx %q2, %q3;` | second logical CNOT on the other pair |
| `mz %m0, %q0;` | measure q0 in Z, result into m0 (same for the other three) |

That is the entire step: the program exists and prints. Next step
(after review): hand it to the QLX compiler and look at what comes out.

## Step 2 — hand it to the QLX compiler

(Kernel: **QLX (container)** — `import qlx` only exists there.)

First, parsing. `qlx.parse` turns the `.entry` text into an MLIR module;
we then splice in the `fabric.code` declaration so the compiler knows
what a logical qubit *is* (a d=3 surface-code patch). Printing the module
shows the program exactly as the compiler sees it — every `pz`/`cx`/`mz`
is now a typed IR operation.

In [2]:
import qlx
from qlx import ir as mlir_ir

base = qlx.parse(PROGRAM)
inner = str(base).strip().removeprefix("module {").removesuffix("}").strip()
module = mlir_ir.Module.parse(
    "module {\n" + CODE_DECLARATION + "\n" + inner + "\n}", base.context)
print(module)

module {
  fabric.code @surface_demo {distance = 3 : i64, partitions = {data = 9 : i64, sx = 4 : i64, sz = 4 : i64}, stabilize_rounds = 3 : i64}
  qlx.target "qlx-c"
  qlx.region_decl @Cs0 {code = "surface_demo", role = #qlx.role<compute>}
  qlx.entry @surface_demo_entry(%arg0: !qlx.region) {
    %0 = qlx.pz %arg0 : !qlx.lqbit
    %1 = qlx.pz %arg0 : !qlx.lqbit
    %2 = qlx.pz %arg0 : !qlx.lqbit
    %3 = qlx.pz %arg0 : !qlx.lqbit
    %ctrl_out, %targ_out = qlx.cx %0, %1, %arg0 : !qlx.lqbit, !qlx.lqbit
    %ctrl_out_0, %targ_out_1 = qlx.cx %2, %3, %arg0 : !qlx.lqbit, !qlx.lqbit
    %4 = qlx.mz %ctrl_out, %arg0 : !qlx.lbit
    %5 = qlx.mz %targ_out, %arg0 : !qlx.lbit
    %6 = qlx.mz %ctrl_out_0, %arg0 : !qlx.lbit
    %7 = qlx.mz %targ_out_1, %arg0 : !qlx.lbit
    qlx.ret
  } attributes {region_params = [@Cs0]}
}



Now the actual compile. `StimViaTQEC` runs the pipeline
*qlx.entry → block graph → TQEC → stim circuit*: each abstract `cx`
becomes a concrete lattice-surgery merge/split laid out in spacetime,
with every physical qubit, stabilizer measurement, and detector
spelled out.

One honest wrinkle: QLX (alpha) has a bug in this lowering. It places
each measurement's end-block at a global `[0, 0, t]` spacetime slot
instead of on the measured qubit's own patch column, so any program
with **more than one measurement** produces a broken block graph and
TQEC rejects it (*"pipe must connect two nearby cubes in direction Z"*).
The repair below is purely geometric: a time-direction pipe must be
vertical, so each floating end-block (it hangs off exactly one pipe)
is snapped onto the column of the block directly below it. On a program
QLX gets right (single measurement) the repair provably changes
nothing — we checked that on the 1-measurement variant of this
program.

In [3]:
import json
from collections import Counter

import stim


def straighten(graph):
    """Snap floating end-blocks onto their qubit column.

    QLX alpha mis-places measurement blocks at a global [0, 0, t] slot;
    every time-direction ("...O") pipe must be vertical, so move each
    dangling end-block (degree 1) to sit directly above its partner.
    """
    cubes = {tuple(c["position"]): c for c in graph["cubes"]}
    degree = Counter()
    for p in graph["pipes"]:
        degree[tuple(p["u"])] += 1
        degree[tuple(p["v"])] += 1
    for p in graph["pipes"]:
        if not p["kind"].endswith("O"):
            continue
        u, v = tuple(p["u"]), tuple(p["v"])
        above_u = (u[0], u[1], u[2] + 1)
        if v != above_u:
            assert degree[v] == 1 and above_u not in cubes
            cube = cubes.pop(v)
            cube["position"] = list(above_u)
            cubes[above_u] = cube
            p["v"] = list(above_u)
    graph["cubes"] = list(cubes.values())
    return graph


class StimViaTQECRepaired(qlx.StimViaTQEC):
    """StimViaTQEC with the block graph straightened before TQEC."""

    def post_process(self, tqec_json):
        graph = straighten(json.loads(tqec_json))
        return super().post_process(json.dumps(graph))


emission = qlx.Assembler(module).emit(StimViaTQECRepaired(k=1))
circuit = stim.Circuit(emission.text)

layers = {int(c[-1]) for c in circuit.get_detector_coordinates().values()}
print(f"physical qubits : {circuit.num_qubits}")
print(f"detectors       : {circuit.num_detectors}  "
      f"(across {len(layers)} time layers -> {len(layers)} syndrome rounds)")
print(f"observables     : {circuit.num_observables}")
print(f"measurements    : {circuit.num_measurements}")
print(f"gate kinds      : {sorted({inst.name for inst in circuit.flattened()})}")
print()
print("--- first 25 lines of the emitted circuit ---")
print("\n".join(str(circuit).splitlines()[:25]))

physical qubits : 162
detectors       : 512  (across 18 time layers -> 18 syndrome rounds)
observables     : 4
measurements    : 594
gate kinds      : ['CX', 'CZ', 'DETECTOR', 'M', 'MX', 'OBSERVABLE_INCLUDE', 'QUBIT_COORDS', 'R', 'RX', 'TICK']

--- first 25 lines of the emitted circuit ---
QUBIT_COORDS(0, 0) 0
QUBIT_COORDS(0, 2) 1
QUBIT_COORDS(0, 4) 2
QUBIT_COORDS(0, 6) 3
QUBIT_COORDS(0, 8) 4
QUBIT_COORDS(0, 10) 5
QUBIT_COORDS(0, 12) 6
QUBIT_COORDS(0, 14) 7
QUBIT_COORDS(1, 1) 8
QUBIT_COORDS(1, 3) 9
QUBIT_COORDS(1, 5) 10
QUBIT_COORDS(1, 7) 11
QUBIT_COORDS(1, 9) 12
QUBIT_COORDS(1, 11) 13
QUBIT_COORDS(1, 13) 14
QUBIT_COORDS(2, 0) 15
QUBIT_COORDS(2, 2) 16
QUBIT_COORDS(2, 4) 17
QUBIT_COORDS(2, 6) 18
QUBIT_COORDS(2, 8) 19
QUBIT_COORDS(2, 10) 20
QUBIT_COORDS(2, 12) 21
QUBIT_COORDS(2, 14) 22
QUBIT_COORDS(3, 1) 23
QUBIT_COORDS(3, 3) 24


What to notice:

- our program became a **circuit-level** object: 162 physical qubits,
  512 detectors over 18 time layers, real `CX`/`CZ` data–ancilla
  coupling — the two lattice-surgery CNOTs, unrolled;
- **4 observables** — one logical outcome per logical qubit;
- QLX schedules the second CNOT *after* the first (q2/q3's patches are
  even initialized two time slots later), so the circuit is deeper than
  a parallel schedule would be;
- the circuit is **noiseless** by design — noise is a separate,
  deliberate choice we make next.

Next step (after review): add a noise model and check the circuit is a
real decoding problem (its error model couples syndromes to the logical
observables).